In [ ]:
import os
import sys
import logging

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

os.environ["CUDA_VISIBLE_DEVICES"] = "3"
#os.environ["TRITON_DEBUG"] = "1"
#os.environ["FLASH_ATTENTION_TRITON_AMD_DEBUG"] = "1"
os.environ["FLASH_ATTENTION_TRITON_AMD_AUTOTUNE"] = os.environ["TRITON_PRINT_AUTOTUNING "] = "1"

import math
import time
from datetime import timedelta
from tqdm import tqdm
import torch

from transformers.modeling_flash_attention_utils import _flash_attention_forward
from transformers.triton_flash_attention_fp8 import attention_backward_triton_impl
from torchtitan.logging import init_logger, logger

model_type = "llama2-7b"
log_step = 10
max_step = 10
exclude_input_cvt = False
use_fp8 = False
eval_bwd = False

torch_dtype = torch.bfloat16
e4m3_dtype = torch.float8_e4m3fnuz
e5m2_dtype = torch.float8_e5m2fnuz
device = torch.device("cuda")

def prepare_data(model_type: str, device: torch.device):
    torch.manual_seed(0)

    configs = {
        "opt-125m": {
            "seqlen": 2048,
            "n_head": 12,
            "n_head_kv": 12,
            "head_dim": 64,
        },
        "llama2-7b": {
            "seqlen": 4096,
            "n_head": 32,
            "n_head_kv": 32,
            "head_dim": 128,
        },
        "llama2-70b": {
            "seqlen": 4096,
            "n_head": 64,
            "n_head_kv": 64,
            "head_dim": 128,
        },
        "llama3-8b": {
            "seqlen": 8192,
            "n_head": 32,
            "n_head_kv": 8,
            "head_dim": 128,
        },
        "llama3-70b": {
            "seqlen": 8192,
            "n_head": 64,
            "n_head_kv": 8,
            "head_dim": 128,
        },
    }

    c=configs[model_type]
    seqlen = c["seqlen"]
    n_head = c["n_head"]
    n_head_kv = c["n_head_kv"]
    head_dim = c["head_dim"]
    query_states = torch.randn((1, seqlen, n_head, head_dim),
                               dtype=torch_dtype,
                               device=device)
    key_states = torch.randn((1, seqlen, n_head_kv, head_dim),
                             dtype=torch_dtype,
                             device=device)
    value_states = torch.randn((1, seqlen, n_head_kv, head_dim),
                               dtype=torch_dtype,
                               device=device)
    output_states = torch.randn((1, seqlen, n_head, head_dim),
                                dtype=torch_dtype,
                                device=device)
    softmax_lse = torch.randn((1, n_head, seqlen),
                                  device=device,
                                  dtype=torch.float32)
    sm_scale = 1 / math.sqrt(head_dim)
    return query_states, key_states, value_states, output_states, softmax_lse, sm_scale


def check_and_convert_fp8(t, descale):
    finfo = torch.finfo(e4m3_dtype)
    return ((t * descale).clamp(min=finfo.min, max=finfo.max).to(e4m3_dtype)
            if t.dtype != e4m3_dtype else t)

def backward(do, q, k, v, o, softmax_lse, fp8_scales):
    q_scale, k_scale, v_scale, p_scale, _ = fp8_scales
    use_fp8 = q_scale is not None
    if use_fp8:
        float8_fw = e4m3_dtype
        float8_bw = e5m2_dtype

        def check_and_convert(t, scale):
            finfo = torch.finfo(float8_bw)
            return ((t * scale).clamp(min=finfo.min, max=finfo.max).to(
                dtype=float8_bw)
                    if t.dtype not in [float8_bw, float8_fw] else t)

        do_scale = torch.finfo(float8_bw).max / do.max()

        # TODO: determine o_scale outside this function
        o_scale = torch.finfo(float8_bw).max / o.max()

        do = check_and_convert(do, do_scale)
        q = check_and_convert(q, q_scale)
        k = check_and_convert(k, k_scale)
        v = check_and_convert(v, v_scale)
        o = check_and_convert(o, o_scale)
    else:
        do_scale = torch.tensor([1.], device=q.device)
        q_scale = torch.tensor([1.], device=q.device)
        k_scale = torch.tensor([1.], device=q.device)
        v_scale = torch.tensor([1.], device=q.device)
        o_scale = torch.tensor([1.], device=q.device)
        p_scale = torch.tensor([1.], device=q.device)

    din = attention_backward_triton_impl(
        do,
        q,
        k,
        v,
        o,
        q_scale,
        k_scale,
        v_scale,
        p_scale,
        o_scale,
        do_scale,
        softmax_lse,
        None,
        None,
        None,
        sm_scale,
        None,
        True,
        "bshd",
        None,
        None,
        None,
        None,
        True,
        use_fp8,
        sequence_parallel=True,
    )
    return din

query_states, key_states, value_states, output_states, softmax_lse, sm_scale = prepare_data(model_type, device)

if eval_bwd:
    loss_fn = torch.nn.MSELoss()
    query_states = torch.nn.Parameter(query_states, requires_grad=True)
    key_states = torch.nn.Parameter(key_states, requires_grad=True)
    value_states = torch.nn.Parameter(value_states, requires_grad=True)

range_q = range_k = range_v = range_o = 16
range_p = 1
num_tokens = query_states.shape[0] * query_states.shape[1]
query_length = query_states.shape[1]

descale_q = descale_k = descale_v = None
if use_fp8:
    dtype_max = torch.finfo(e4m3_dtype).max

    descale_q = torch.tensor(dtype_max / range_q, device=device)
    descale_k = torch.tensor(dtype_max / range_k, device=device)
    descale_v = torch.tensor(dtype_max / range_v, device=device)
    descale_p = torch.scalar_tensor(240.0, device=query_states.device)
    descale_o = torch.scalar_tensor(1.0, device=query_states.device)
    fp8_scales = (descale_q, descale_k, descale_v, descale_p, descale_o)
    if exclude_input_cvt:
        query_states = check_and_convert_fp8(query_states, descale_q)
        key_states = check_and_convert_fp8(key_states, descale_k)
        value_states = check_and_convert_fp8(value_states, descale_v)
else:
    fp8_scales = (None, None, None, None, None)
print(f"descale_q={descale_q}, descale_k={descale_k}, descale_v={descale_v}")
fa_output = _flash_attention_forward(
    query_states,
    key_states,
    value_states,
    None,
    query_length,
    True,
    0.0,
    softmax_scale=sm_scale,
    descale_q=descale_q,
    descale_k=descale_k,
    descale_v=descale_v,
)
print(fa_output.dtype)
if eval_bwd:
    #if use_fp8 and exclude_input_cvt:
    if True:
        backward(output_states, query_states, key_states, value_states,
                 fa_output, softmax_lse, fp8_scales)
    else:

        loss = loss_fn(fa_output, output_states)
        loss.backward()
torch.cuda.synchronize()
time_last_log = time.perf_counter()
tps_list = []

for i in tqdm(range(max_step)):
    fa_output = _flash_attention_forward(
        query_states,
        key_states,
        value_states,
        None,
        query_length,
        True,
        0.0,
        softmax_scale=sm_scale,
        descale_q=descale_q,
        descale_k=descale_k,
        descale_v=descale_v,
    )
    if eval_bwd:
        #if use_fp8 and exclude_input_cvt:
        if True:
            backward(output_states, query_states, key_states, value_states,
                    fa_output, softmax_lse, fp8_scales)
        else:
            loss = loss_fn(fa_output, output_states)
            loss.backward()
    torch.cuda.synchronize()

    time_delta = time.perf_counter() - time_last_log
    # tokens per second, abbreviated as tps
    tps = num_tokens / time_delta
    tps_list.append(tps)
    # if i % log_step == 0:
    #     logger.info(f"tps: {tps}")
    time_last_log = time.perf_counter()

logger.info(f"mean tps: {torch.tensor(tps_list).mean()}")


descale_q=None, descale_k=None, descale_v=None
torch.bfloat16


100%|██████████| 10/10 [00:00<00:00, 294.50it/s]

INFO:root:mean tps: 1163394.625


In [2]:
from transformers.triton_flash_attention_fp8 import attn_fwd, _bwd_kernel

kernel = list(_bwd_kernel.fn.device_caches[0][0].values())[0]
#print(print(_bwd_kernel.fn.device_caches[0]))
# print(print(
#     list(_bwd_kernel.fn.device_caches[0][0].values())[0].asm['amdgcn']))
# # _bwd_kernel.fn.compile()
print(kernel.asm.keys())

dict_keys(['ttir', 'ttgir', 'llir', 'amdgcn', 'hsaco'])


In [3]:
print(kernel.asm['amdgcn'])

	.text
	.amdgcn_target "amdgcn-amd-amdhsa--gfx942"
	.amdhsa_code_object_version 5
	.globl	_bwd_kernel                     ; -- Begin function _bwd_kernel
	.p2align	8
	.type	_bwd_kernel,@function
_bwd_kernel:                            ; @_bwd_kernel
.Lfunc_begin0:
	.cfi_sections .debug_frame
	.cfi_startproc
	s_trap 2 ; Kernarg preload header. Trap with incompatible firmware that doesn't support preloading kernel arguments.
	.fill 63, 4, 0xbf800000 ; s_nop 0
; %bb.0:
	.file	1 "/workspace/transformers/src/transformers" "triton_flash_attention_fp8.py"
	.loc	1 1770 27 prologue_end          ; triton_flash_attention_fp8.py:1770:27
	s_load_dwordx4 s[44:47], s[0:1], 0x90
	s_load_dwordx2 s[60:61], s[0:1], 0xc0
	.loc	1 1773 22                       ; triton_flash_attention_fp8.py:1773:22
	s_mul_hi_i32 s10, s14, 0x2aaaaaab
	s_lshr_b32 s11, s10, 31
	s_ashr_i32 s19, s10, 1
	s_add_i32 s19, s19, s11
	.loc	1 1803 27                       ; triton_flash_attention_fp8.py:1803:27
	s_waitcnt lgkmcnt(0)
	s